# ReLU Tessellation Dynamics: Standard vs Adversarial Training

This notebook runs the full experimental pipeline on Google Colab with GPU support.

**Requirements:** Select a **GPU runtime** before running (Runtime > Change runtime type > T4 GPU).

The setup installs:
- `graph-tool` via conda (required by SplineCam for exact tessellation computation)
- SplineCam from the official repository
- All other Python dependencies (torch, numpy, scipy, matplotlib, etc.)

## 0. Verify GPU availability

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > T4 GPU, "
        "then restart the runtime."
    )

## 1. Install graph-tool via conda (matched to kernel Python)

`graph-tool` is a C++ library not available via pip. The Ubuntu apt package is built for Python 3.10, but Colab's kernel runs Python 3.12 — the compiled extensions are ABI-incompatible across minor versions.

We install miniconda and use conda-forge to get a graph-tool build that matches the kernel's exact Python version. This takes 3-5 minutes.

In [ ]:
import sys
PY_VER = f"{sys.version_info.major}.{sys.version_info.minor}"
print(f"Colab kernel Python: {PY_VER} ({sys.executable})")
print(f"We will install graph-tool for Python {PY_VER} via conda-forge.")

## 2. Install miniconda, graph-tool, and fix system libraries

The next two cells handle:
1. **Cell A**: Installs miniconda + graph-tool via conda-forge (skip if already done)
2. **Cell B**: Copies the conda env's newer `libstdc++` over the system version and **restarts the runtime** (only on first run — it detects if already done)
3. **Cell C**: Adds the conda env to `sys.path` and imports `graph_tool`

After the restart, re-run all cells from the top. On second run, Cell B will skip the restart and Cell C will import normally.

In [ ]:
import os, sys

CONDA_PREFIX = "/opt/conda"
CONDA = f"{CONDA_PREFIX}/bin/conda"
PY_VER = f"{sys.version_info.major}.{sys.version_info.minor}"  # Colab kernel version
GT_ENV = f"{CONDA_PREFIX}/envs/gt"

# Step 1: Install miniconda if not present
if not os.path.exists(CONDA):
    print("=== Installing miniconda ===")
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    !bash /tmp/miniconda.sh -b -p {CONDA_PREFIX} 2>&1 | tail -2
    !rm /tmp/miniconda.sh

# Step 2: Configure conda — remove defaults entirely, use only conda-forge
!{CONDA} config --remove channels defaults 2>/dev/null; true
!{CONDA} config --add channels conda-forge 2>/dev/null; true
!{CONDA} config --set channel_priority strict
print("Conda channels configured (conda-forge only)")

# Step 3: Create env with --override-channels to fully bypass any default/TOS channels
if not os.path.exists(GT_ENV):
    print(f"\n=== Creating conda env 'gt' with Python {PY_VER} + graph-tool ===")
    print("(This takes 3-5 minutes...)")
    !{CONDA} create -y --override-channels -c conda-forge -n gt python={PY_VER} graph-tool 2>&1 | tail -15
else:
    print(f"Conda env 'gt' already exists at {GT_ENV}")

# Step 4: Verify
print("\n=== Verification ===")
gt_python = f"{GT_ENV}/bin/python"
if os.path.exists(gt_python):
    !{gt_python} --version
    !{gt_python} -c "import graph_tool; print('graph-tool', graph_tool.__version__)"
else:
    print(f"ERROR: {gt_python} not found — env creation may have failed.")
    print("Check the output above for errors.")

In [ ]:
import os, sys, shutil, subprocess

GT_ENV = "/opt/conda/envs/gt"
conda_lib = f"{GT_ENV}/lib"
dst_dir = "/lib/x86_64-linux-gnu"
src = os.path.realpath(f"{conda_lib}/libstdc++.so.6")
dst = f"{dst_dir}/libstdc++.so.6"

# Check if the system libstdc++ already has CXXABI_1.3.15
result = subprocess.run(["strings", dst], capture_output=True, text=True)
if "CXXABI_1.3.15" in result.stdout:
    print("System libstdc++ already has CXXABI_1.3.15 — no restart needed.")
else:
    print(f"Copying {src} -> {dst}")
    shutil.copy2(src, dst)
    # Verify the copy worked
    result2 = subprocess.run(["strings", dst], capture_output=True, text=True)
    assert "CXXABI_1.3.15" in result2.stdout, "Copy failed — CXXABI_1.3.15 still missing"
    print("Library updated. Restarting runtime to load the new version...")
    print(">>> After restart, re-run from Cell 0 (GPU check) through here, then continue. <<<")
    os.kill(os.getpid(), 9)  # Force kernel restart

In [ ]:
import sys, os

PY_VER = f"{sys.version_info.major}.{sys.version_info.minor}"
GT_ENV = "/opt/conda/envs/gt"
conda_sp = f"{GT_ENV}/lib/python{PY_VER}/site-packages"
conda_lib = f"{GT_ENV}/lib"

# Add conda env's site-packages and libraries to the search path
if conda_sp not in sys.path:
    sys.path.insert(0, conda_sp)
ld_path = os.environ.get("LD_LIBRARY_PATH", "")
if conda_lib not in ld_path:
    os.environ["LD_LIBRARY_PATH"] = f"{conda_lib}:{ld_path}"

print(f"Python: {PY_VER}")
print(f"conda site-packages: {conda_sp}")

import graph_tool
print(f"\ngraph-tool version: {graph_tool.__version__}")
print(f"Location: {graph_tool.__file__}")

# Install remaining pip packages into Colab's default Python
!pip install -q torch torchvision matplotlib numpy scipy seaborn tqdm \
    python-igraph networkx livelossplot

import torch
print(f"\nPyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

## 3. Clone repositories and install SplineCam

In [ ]:
import os, sys, re

# Clone SplineCam
SPLINECAM_DIR = "/content/splinecam"
if not os.path.exists(SPLINECAM_DIR):
    !git clone https://github.com/AhmedImtiazPrio/splinecam.git {SPLINECAM_DIR}
else:
    print(f"SplineCam already cloned at {SPLINECAM_DIR}")

# --- Patch SplineCam for Python 3.12 + modern PyTorch compatibility ---

def annotate_def_params(line):
    """Add type annotations to unannotated default params in a def line."""
    match = re.match(r'(.*def \w+\()(.*)(\):.*)', line)
    if not match:
        return line
    prefix, params_str, suffix = match.groups()
    new_params = []
    for param in params_str.split(','):
        param = param.strip()
        if not param or '=' not in param:
            new_params.append(param)
            continue
        name_part, val_part = param.split('=', 1)
        name_part = name_part.strip()
        val_part = val_part.strip()
        if ':' in name_part:
            new_params.append(f"{name_part}={val_part}")
            continue
        if re.match(r'^-?\d+e[+-]?\d+$', val_part) or re.match(r'^-?\d+\.\d+$', val_part):
            new_params.append(f"{name_part}: float={val_part}")
        elif re.match(r'^-?\d+$', val_part):
            if 'pad_dist' in name_part:
                new_params.append(f"{name_part}: float={val_part}.0")
            else:
                new_params.append(f"{name_part}: int={val_part}")
        else:
            new_params.append(f"{name_part}={val_part}")
    return prefix + ', '.join(new_params) + suffix + '\n'

# ── Patch 1: utils.py — TorchScript type annotations ──
utils_path = os.path.join(SPLINECAM_DIR, "splinecam", "utils.py")
with open(utils_path, "r") as f:
    lines = f.readlines()
original = ''.join(lines)
patched_lines = []
for line in lines:
    if line.strip().startswith("def ") and "=" in line:
        line = annotate_def_params(line)
    patched_lines.append(line)
patched = ''.join(patched_lines)
if patched != original:
    with open(utils_path, "w") as f:
        f.write(patched)
    print("Patched utils.py: TorchScript type annotations")
else:
    print("utils.py: already patched")

# ── Patch 2: graph.py — escape sequence + len() on 0-D tensors ──
graph_path = os.path.join(SPLINECAM_DIR, "splinecam", "graph.py")
with open(graph_path, "r") as f:
    content = f.read()
orig_graph = content

# Fix invalid escape sequence \i (Python 3.12 SyntaxWarning)
if "\\i" in content and "\\\\i" not in content:
    content = content.replace("\\i", "\\\\i")

# Fix len() calls on tensors that may be 0-dimensional.
# Instead of a try/except wrapper (TorchScript doesn't support try),
# directly replace len(var) with var.shape[0] for tensor variables.
# .shape[0] works on 1-D+ tensors and raises IndexError (not TypeError)
# on 0-D tensors, but the real fix is to ensure these are always 1-D
# by reshaping the torch.where output. We use .reshape(-1) approach:
# Replace the torch.where patterns to always produce 1-D output.
if '_patched_len' not in content:
    # Replace specific len() calls with .shape[0] which is safer
    # But better: wrap torch.where results with .reshape(-1) to guarantee 1-D
    content = content.replace(
        'no_inter_idx = torch.where(torch.prod(q[:-1] == q[1:],axis=1))[0].cpu()',
        'no_inter_idx = torch.where(torch.prod(q[:-1] == q[1:],axis=1))[0].reshape(-1).cpu()'
    )
    content = content.replace(
        'inter_idx = torch.where(~torch.prod(q[:-1] == q[1:],axis=1))[0].cpu()',
        'inter_idx = torch.where(~torch.prod(q[:-1] == q[1:],axis=1))[0].reshape(-1).cpu()'
    )
    # Also fix any other torch.where()[0] patterns to add .reshape(-1)
    content = re.sub(
        r'torch\.where\(([^)]+)\)\[0\](?!\.reshape)',
        r'torch.where(\1)[0].reshape(-1)',
        content
    )
    # Mark as patched
    content = content.replace(
        'import torch\n',
        'import torch\n_patched_len = True\n',
        1
    )

if content != orig_graph:
    with open(graph_path, "w") as f:
        f.write(content)
    print("Patched graph.py: escape sequence + reshape(-1) for torch.where outputs")
else:
    print("graph.py: already patched")

# ── Patch 3: compute.py — disable multiprocessing ──
compute_path = os.path.join(SPLINECAM_DIR, "splinecam", "compute.py")
with open(compute_path, "r") as f:
    content = f.read()
if "num_workers=n_workers" in content:
    content = content.replace("num_workers=n_workers", "num_workers=0")
    with open(compute_path, "w") as f:
        f.write(content)
    print("Patched compute.py: set DataLoader num_workers=0")

# Add to sys.path
if SPLINECAM_DIR not in sys.path:
    sys.path.insert(0, SPLINECAM_DIR)

# Install SplineCam's dependencies
!pip install -q python-igraph networkx

# Verify
import splinecam
print(f"\nSplineCam imported successfully from {splinecam.__file__}")

In [ ]:
import os

PROJECT_DIR = "/content/adversarial-tesselation-analysis"
if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/BryanZhang938/adversarial-tesselation-analysis.git {PROJECT_DIR}
else:
    print(f"Project already cloned at {PROJECT_DIR}")

assert os.path.exists(PROJECT_DIR), f"Project not found at {PROJECT_DIR}"
print(f"Project directory: {PROJECT_DIR}")
print(os.listdir(PROJECT_DIR))

# ── Patch tessellation_analysis.py: fix SplineCam API call ──
# The function compute_splinecam_tessellation needs to pass 3 separate args
# to get_partitions_with_db(domain_tensor, T_projection, NN_wrapper)
tess_path = os.path.join(PROJECT_DIR, "src", "tessellation_analysis.py")
with open(tess_path, "r") as f:
    content = f.read()

needs_patch = False

# Fix 1: model_wrapper must use as_sequential=False
if "as_sequential=False" not in content and "model_wrapper(" in content:
    content = content.replace(
        "device=device,\n    )",
        "device=device,\n        as_sequential=False,\n    )"
    )
    needs_patch = True

# Fix 2: get_partitions_with_db needs (domain_tensor, T_proj, NN) not (domain, T, None)
if "regions, db_edges = splinecam.compute.get_partitions_with_db(" in content:
    old_call = '''    # Compute partitions with decision boundary
    # Third arg is projection matrix (None for 2D)
    regions, db_edges = splinecam.compute.get_partitions_with_db(
        domain, T, None
    )

    return regions, db_edges, T'''
    new_call = '''    # get_partitions_with_db(domain, T, NN) expects:
    #   domain: (V, 2) tensor of domain vertices on GPU
    #   T: (2, 3) projection matrix [I | 0] for 2D inputs
    #   NN: the model_wrapper object
    domain_t = torch.from_numpy(domain).to(device)
    T_proj = torch.eye(3, dtype=torch.float64, device=device)[:2]  # 2x3: [I | 0]

    regions, db_edges, Abw = splinecam.compute.get_partitions_with_db(
        domain_t, T_proj, NN,
        fwd_batch_size=512, batch_size=64, n_workers=0
    )

    return regions, db_edges, NN'''
    if old_call in content:
        content = content.replace(old_call, new_call)
        needs_patch = True

# Fix 3: Rename T variable to NN in model_wrapper call
if "T = splinecam.wrappers.model_wrapper(" in content:
    content = content.replace(
        "T = splinecam.wrappers.model_wrapper(",
        "NN = splinecam.wrappers.model_wrapper("
    )
    needs_patch = True

if needs_patch:
    with open(tess_path, "w") as f:
        f.write(content)
    print("Patched tessellation_analysis.py: fixed SplineCam API call")
else:
    print("tessellation_analysis.py: already correct")

## 4. Patch SplineCam for Colab compatibility

SplineCam has two known issues in Colab:
1. **TorchScript float typing**: `utils.py` uses integer defaults (e.g. `pad_dist=1`) which TorchScript rejects. We patch them to floats.
2. **Multiprocessing + CUDA**: `compute.py` spawns workers that try to initialize CUDA, causing errors. We patch DataLoader `num_workers` to 0.

In [ ]:
import importlib
import torch

# ── Monkey-patch torch.Tensor.__len__ to handle 0-D tensors ──
# SplineCam calls len() on tensors from torch.where() which can be 0-D.
# Standard PyTorch raises "len() of unsized object" for 0-D tensors.
# This patch makes 0-D tensors return 1 (they contain one element).
_orig_tensor_len = torch.Tensor.__len__
def _patched_tensor_len(self):
    if self.dim() == 0:
        return 1
    return _orig_tensor_len(self)
torch.Tensor.__len__ = _patched_tensor_len
print("Patched torch.Tensor.__len__ to handle 0-D tensors")

# Reload SplineCam modules to pick up file patches
import splinecam.utils
import splinecam.compute
importlib.reload(splinecam.utils)
importlib.reload(splinecam.compute)
# Note: graph.py is NOT reloaded here since it triggers TorchScript
# compilation errors. The file-level patches are picked up on first import.
print("SplineCam modules reloaded with all patches applied.")

## 5. Verify SplineCam works end-to-end

Quick smoke test: wrap a tiny MLP and compute partitions on a small domain.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from splinecam.wrappers import model_wrapper
from splinecam.compute import get_partitions_with_db

# Build a tiny test model
test_model = nn.Sequential(
    nn.Linear(2, 10),
    nn.ReLU(),
    nn.Linear(10, 2),
).eval().cuda()

# Define square domain
domain = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]], dtype=np.float64)
domain_t = torch.from_numpy(domain).cuda()

# Wrap model — as_sequential=False keeps layers as a list,
# which is required because get_partitions_with_db uses slicing (NN.layers[:n])
NN = model_wrapper(test_model, input_shape=(2,), T=None,
                   dtype=torch.float64, device='cuda',
                   as_sequential=False)

# Compute tessellation
T_proj = torch.eye(3, dtype=torch.float64, device='cuda')[:2]  # 2x3: [I | 0]
regions, endpoints, Abw = get_partitions_with_db(
    domain_t, T_proj, NN,
    fwd_batch_size=512, batch_size=64, n_workers=0
)

print(f"Tessellation computed successfully!")
print(f"  Number of regions: {len(regions)}")
print(f"  Endpoints type: {type(endpoints)}")
del test_model, NN, regions, endpoints, Abw
torch.cuda.empty_cache()

## 6. Configure and run experiment

**Reviewer feedback (March 2026):** The PGD-AT model must converge to comparable clean
accuracy as the ERM model for tessellation comparisons to be meaningful.

Strategy:
1. **Increase capacity** for adversarial model (wider layers, e.g. 100–200 neurons)
2. **Train longer** for adversarial model (2–3× the ERM epochs)
3. **Reduce ε** if needed — find the largest ε where the model still converges
4. **Adjust learning rate** — higher initial LR (0.1) with cosine decay for PGD-AT
5. **Track convergence** — plot clean test accuracy and robust test accuracy

We first run a convergence sweep to find good adversarial hyperparameters,
then run the final tessellation comparison with matched clean accuracy.

In [ ]:
import sys
import importlib
sys.path.insert(0, PROJECT_DIR)

import copy
import json
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from configs.experiment_config import (
    ExperimentConfig, _compute_checkpoint_epochs, make_config,
)
from src.datasets import get_dataset, get_dataloader
from src.models import make_relu_mlp, count_parameters
import src.train
importlib.reload(src.train)
from src.train import train_model, evaluate_accuracy, evaluate_robust_accuracy
import src.tessellation_analysis
importlib.reload(src.tessellation_analysis)
from src.tessellation_analysis import analyze_checkpoint
from src.visualization import (
    plot_dataset, plot_training_comparison,
    plot_epoch_snapshots, plot_boundary_distance_histograms,
)

print("All modules imported successfully.")

In [ ]:
# =====================================================
# EXPERIMENT CONFIGURATION
# =====================================================
DATASET = "spirals"
SEED = 42

# Standard (ERM) model — same as before
STD_HIDDEN_DIMS = [50]
STD_EPOCHS = 500
STD_LR = 0.01

# Adversarial (PGD-AT) model — updated per reviewer feedback:
#  - Wider network (more capacity to learn robust features)
#  - Longer training (robust objective converges slower)
#  - Higher initial LR with cosine decay
#  - Slightly reduced epsilon to ensure convergence
ADV_HIDDEN_DIMS = [200]       # wider network for adversarial training
ADV_EPOCHS = 1500             # 3x longer training
ADV_LR = 0.1                  # higher initial LR
ADV_SCHEDULER = "cosine"      # cosine annealing to 0
ADV_EPSILON = 0.05            # reduced epsilon (start conservative)

# Output directories
config_base = ExperimentConfig()
config_base.checkpoint_dir = f"/content/checkpoints_{DATASET}"
config_base.figure_dir = f"/content/figures_{DATASET}"
config_base.results_dir = f"/content/results_{DATASET}"

os.makedirs(config_base.checkpoint_dir, exist_ok=True)
os.makedirs(config_base.figure_dir, exist_ok=True)
os.makedirs(config_base.results_dir, exist_ok=True)

print(f"Dataset: {DATASET}")
print(f"\nStandard model:     hidden={STD_HIDDEN_DIMS}, epochs={STD_EPOCHS}, lr={STD_LR}")
print(f"Adversarial model:  hidden={ADV_HIDDEN_DIMS}, epochs={ADV_EPOCHS}, lr={ADV_LR}, "
      f"scheduler={ADV_SCHEDULER}, eps={ADV_EPSILON}")

## 7. Generate dataset

In [ ]:
dataset, X_train, y_train, X_test, y_test = get_dataset(
    DATASET,
    n_samples=500,
    noise=0.1,
    seed=SEED,
    n_test=200,
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train range: x=[{X_train[:,0].min():.3f}, {X_train[:,0].max():.3f}], "
      f"y=[{X_train[:,1].min():.3f}, {X_train[:,1].max():.3f}]")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_dataset(X_train, y_train, title=f"{DATASET} (train, n=500)", ax=axes[0])
plot_dataset(X_test, y_test, title=f"{DATASET} (test, n=200)", ax=axes[1])
plt.tight_layout()
plt.show()

## 8. Train models (Standard ERM + PGD-AT with convergence tracking)

Key changes from v1:
- **Standard (ERM):** Same as before — [50]-wide MLP, 500 epochs, SGD lr=0.01
- **Adversarial (PGD-AT):** Wider network ([200]), 3x longer (1500 epochs),
  higher LR (0.1) with cosine decay, reduced epsilon (0.05)
- **Both** now track clean test accuracy and robust test accuracy every 10 epochs

In [ ]:
def run_training(config, X_train, y_train, dataset, run_name, X_test, y_test):
    """Train one model and return history."""
    torch.manual_seed(config.seed)
    np.random.seed(config.seed)

    model = make_relu_mlp(
        input_dim=config.model.input_dim,
        hidden_dims=config.model.hidden_dims,
        output_dim=config.model.output_dim,
    )
    print(f"\n{'='*60}")
    print(f"Training: {run_name}")
    print(f"  Architecture: {config.model.hidden_dims}")
    print(f"  Parameters: {count_parameters(model)}")
    print(f"  Epochs: {config.train.epochs}, LR: {config.train.lr}, "
          f"Scheduler: {config.train.scheduler}")
    if config.adv.enabled:
        print(f"  Adversarial: eps={config.adv.epsilon}, "
              f"K={config.adv.num_steps}, alpha={config.adv.step_size:.4f}")
    print(f"{'='*60}")

    dataloader = get_dataloader(dataset, batch_size=config.train.batch_size)
    checkpoint_dir = os.path.join(config.checkpoint_dir, run_name)

    history = train_model(
        model, dataloader, config,
        checkpoint_dir=checkpoint_dir,
        run_name=run_name,
        device=config.device,
        X_test=X_test,
        y_test=y_test,
    )
    return history, checkpoint_dir

In [ ]:
# --- Standard (ERM) training ---
config_std = make_config(
    dataset=DATASET,
    hidden_dims=STD_HIDDEN_DIMS,
    epochs=STD_EPOCHS,
    lr=STD_LR,
    scheduler="none",
    adversarial=False,
    seed=SEED,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
config_std.checkpoint_dir = config_base.checkpoint_dir
config_std.figure_dir = config_base.figure_dir
config_std.results_dir = config_base.results_dir

hist_std, ckpt_dir_std = run_training(
    config_std, X_train, y_train, dataset, "standard", X_test, y_test
)

# Report final test accuracy
final_test_std = evaluate_accuracy(
    make_relu_mlp(2, STD_HIDDEN_DIMS, 2).to(config_std.device),
    X_test, y_test, device=config_std.device
)
# Actually use the trained model's last test accuracy from history
final_test_std = [x for x in hist_std["test_clean_acc"] if x > 0][-1]
print(f"\nStandard model final clean test accuracy: {final_test_std:.4f}")

In [ ]:
# --- Adversarial (PGD-AT) training ---
# Uses wider network, longer training, higher LR with cosine decay
config_adv = make_config(
    dataset=DATASET,
    hidden_dims=ADV_HIDDEN_DIMS,
    epochs=ADV_EPOCHS,
    epsilon=ADV_EPSILON,
    lr=ADV_LR,
    scheduler=ADV_SCHEDULER,
    adversarial=True,
    seed=SEED,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
config_adv.checkpoint_dir = config_base.checkpoint_dir
config_adv.figure_dir = config_base.figure_dir
config_adv.results_dir = config_base.results_dir

hist_adv, ckpt_dir_adv = run_training(
    config_adv, X_train, y_train, dataset, "adversarial", X_test, y_test
)

final_test_adv = [x for x in hist_adv["test_clean_acc"] if x > 0][-1]
final_robust_adv = [x for x in hist_adv["test_robust_acc"] if x > 0][-1] if any(
    x > 0 for x in hist_adv["test_robust_acc"]) else 0.0
print(f"\nAdversarial model final clean test accuracy: {final_test_adv:.4f}")
print(f"Adversarial model final robust test accuracy: {final_robust_adv:.4f}")

In [ ]:
%matplotlib inline

# ---- CONVERGENCE VERIFICATION (reviewer requirement) ----
# Plot clean test accuracy and robust test accuracy over training

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Training loss
ax = axes[0, 0]
ax.plot(hist_std["epoch"], hist_std["train_loss"], label="Standard", color='#2196F3')
ax.plot(hist_adv["epoch"], hist_adv["train_loss"], label="Adversarial", color='#FF5722')
ax.set_xlabel("Epoch"); ax.set_ylabel("Training Loss"); ax.set_title("(a) Training Loss")
ax.legend()

# (b) Training accuracy (clean)
ax = axes[0, 1]
ax.plot(hist_std["epoch"], hist_std["train_acc"], label="Standard", color='#2196F3')
ax.plot(hist_adv["epoch"], hist_adv["train_acc"], label="Adversarial", color='#FF5722')
ax.set_xlabel("Epoch"); ax.set_ylabel("Clean Train Accuracy"); ax.set_title("(b) Clean Train Accuracy")
ax.legend()

# (c) Clean test accuracy — KEY CONVERGENCE CHECK
ax = axes[1, 0]
# Filter to epochs where test was evaluated (non-zero entries)
std_test_epochs = [e for e, a in zip(hist_std["epoch"], hist_std["test_clean_acc"]) if a > 0]
std_test_accs = [a for a in hist_std["test_clean_acc"] if a > 0]
adv_test_epochs = [e for e, a in zip(hist_adv["epoch"], hist_adv["test_clean_acc"]) if a > 0]
adv_test_accs = [a for a in hist_adv["test_clean_acc"] if a > 0]

ax.plot(std_test_epochs, std_test_accs, 'o-', label="Standard (clean)", color='#2196F3', markersize=2)
ax.plot(adv_test_epochs, adv_test_accs, 's-', label="Adversarial (clean)", color='#FF5722', markersize=2)
ax.axhline(y=0.95, color='gray', linestyle='--', alpha=0.5, label='95% threshold')
ax.set_xlabel("Epoch"); ax.set_ylabel("Clean Test Accuracy")
ax.set_title("(c) Clean Test Accuracy (CONVERGENCE CHECK)")
ax.legend(fontsize=9)

# (d) Robust test accuracy (adversarial only)
ax = axes[1, 1]
adv_robust_epochs = [e for e, a in zip(hist_adv["epoch"], hist_adv["test_robust_acc"]) if a > 0]
adv_robust_accs = [a for a in hist_adv["test_robust_acc"] if a > 0]
if adv_robust_epochs:
    ax.plot(adv_robust_epochs, adv_robust_accs, 's-', label="Adversarial (robust)",
            color='#FF5722', markersize=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("Robust Test Accuracy")
ax.set_title("(d) Robust Test Accuracy (PGD-AT only)")
ax.legend(fontsize=9)

plt.suptitle("Convergence Verification: Standard vs Adversarial", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(config_base.figure_dir, "convergence_verification.png"), dpi=200)
plt.show()

# ---- Convergence check ----
gap = abs(final_test_std - final_test_adv)
print(f"\n{'='*60}")
print(f"CONVERGENCE CHECK")
print(f"  Standard clean test acc:     {final_test_std:.4f}")
print(f"  Adversarial clean test acc:  {final_test_adv:.4f}")
print(f"  Gap:                         {gap:.4f}")
if final_robust_adv > 0:
    print(f"  Adversarial robust test acc: {final_robust_adv:.4f}")
if gap <= 0.02:
    print(f"  PASS: Gap <= 2% — models have comparable clean accuracy.")
    print(f"  Tessellation comparison is meaningful.")
else:
    print(f"  WARNING: Gap > 2% — adversarial model has not converged.")
    print(f"  Consider: wider network, more epochs, lower epsilon, or different LR.")
print(f"{'='*60}")

## 9. Compute tessellation statistics at each checkpoint

Now that both models have converged to comparable clean accuracy, we compute
tessellation statistics at M=50 evenly spaced checkpoints for each.

Note: the standard and adversarial models may have different architectures
and different numbers of epochs, so we analyze each with its own config.

In [ ]:
import importlib
import src.tessellation_analysis
importlib.reload(src.tessellation_analysis)
from src.tessellation_analysis import analyze_checkpoint

def analyze_all_checkpoints(config, run_name, ckpt_dir, X_train):
    """Analyze all checkpoints for one training run."""
    model_factory = lambda: make_relu_mlp(
        input_dim=config.model.input_dim,
        hidden_dims=config.model.hidden_dims,
        output_dim=config.model.output_dim,
    )

    all_stats = []
    all_grids = []

    for epoch in config.tess.checkpoint_epochs:
        ckpt_path = os.path.join(ckpt_dir, f"{run_name}_epoch{epoch:04d}.pt")
        if not os.path.exists(ckpt_path):
            print(f"  Checkpoint not found: epoch {epoch}")
            continue

        print(f"  Analyzing epoch {epoch}...", end=" ")
        stats, grid_data = analyze_checkpoint(
            model_factory, ckpt_path, X_train, config, device=config.device
        )
        all_stats.append(stats)
        all_grids.append(grid_data)

        print(f"Regions={stats['num_regions_grid']} | "
              f"LC={stats.get('local_region_count_mean', 0):.1f} | "
              f"DBD={stats.get('boundary_dist_mean', 0):.4f} | "
              f"BD={stats.get('boundary_density', 0):.2f}")

    return all_stats, all_grids

In [ ]:
print("Analyzing STANDARD checkpoints...")
stats_std, grids_std = analyze_all_checkpoints(
    config_std, "standard", ckpt_dir_std, X_train
)

In [ ]:
print("Analyzing ADVERSARIAL checkpoints...")
stats_adv, grids_adv = analyze_all_checkpoints(
    config_adv, "adversarial", ckpt_dir_adv, X_train
)

## 10. Generate paper figures

In [ ]:
%matplotlib inline

# Main metrics comparison
# Note: standard and adversarial may have different epoch ranges now
# We normalize to [0, 1] training progress for fair comparison

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

epochs_s = [s["epoch"] for s in stats_std]
epochs_a = [s["epoch"] for s in stats_adv]
# Normalize to training progress fraction
progress_s = [e / STD_EPOCHS for e in epochs_s]
progress_a = [e / ADV_EPOCHS for e in epochs_a]

# (a) Number of linear regions
ax = axes[0, 0]
ax.plot(progress_s, [s["num_regions_grid"] for s in stats_std],
        'o-', label=f"Standard ({STD_HIDDEN_DIMS})", color='#2196F3', markersize=3)
ax.plot(progress_a, [s["num_regions_grid"] for s in stats_adv],
        's-', label=f"Adversarial ({ADV_HIDDEN_DIMS})", color='#FF5722', markersize=3)
ax.set_xlabel("Training Progress"); ax.set_ylabel("# Linear Regions (grid)")
ax.set_title("(a) Region Count"); ax.legend(fontsize=9)

# (b) Local region density
ax = axes[0, 1]
ax.plot(progress_s, [s.get("local_region_count_mean", 0) for s in stats_std],
        'o-', label="Standard", color='#2196F3', markersize=3)
ax.plot(progress_a, [s.get("local_region_count_mean", 0) for s in stats_adv],
        's-', label="Adversarial", color='#FF5722', markersize=3)
ax.set_xlabel("Training Progress"); ax.set_ylabel("Mean Local Region Count")
ax.set_title("(b) Local Complexity Near Data"); ax.legend(fontsize=9)

# (c) Mean distance to decision boundary
ax = axes[1, 0]
ax.plot(progress_s, [s.get("boundary_dist_mean", 0) for s in stats_std],
        'o-', label="Standard", color='#2196F3', markersize=3)
ax.plot(progress_a, [s.get("boundary_dist_mean", 0) for s in stats_adv],
        's-', label="Adversarial", color='#FF5722', markersize=3)
ax.set_xlabel("Training Progress"); ax.set_ylabel("Mean Boundary Distance")
ax.set_title("(c) Data-to-Boundary Distance"); ax.legend(fontsize=9)

# (d) Boundary density
ax = axes[1, 1]
ax.plot(progress_s, [s.get("boundary_density", 0) for s in stats_std],
        'o-', label="Standard", color='#2196F3', markersize=3)
ax.plot(progress_a, [s.get("boundary_density", 0) for s in stats_adv],
        's-', label="Adversarial", color='#FF5722', markersize=3)
ax.set_xlabel("Training Progress"); ax.set_ylabel("Boundary Density")
ax.set_title("(d) Boundary Density"); ax.legend(fontsize=9)

fig.suptitle(f"Tessellation Statistics: Standard vs. Adversarial\n"
             f"(Std: {STD_HIDDEN_DIMS}, {STD_EPOCHS}ep | "
             f"Adv: {ADV_HIDDEN_DIMS}, {ADV_EPOCHS}ep, eps={ADV_EPSILON})",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(config_base.figure_dir, "metrics_comparison.pdf"))
plt.savefig(os.path.join(config_base.figure_dir, "metrics_comparison.png"), dpi=200)
plt.show()

In [ ]:
# Epoch snapshot visualizations — standard model
vis_indices = [0, len(stats_std)//4, len(stats_std)//2, -1]
vis_grids_std = [grids_std[i] for i in vis_indices]
vis_epochs_std = [stats_std[i]["epoch"] for i in vis_indices]

plot_epoch_snapshots(
    vis_grids_std, X_train, y_train, vis_epochs_std,
    run_name="standard", figure_dir=config_base.figure_dir
)
display(Image(filename=os.path.join(config_base.figure_dir, "snapshots_standard.png")))

In [ ]:
# Epoch snapshot visualizations — adversarial model
vis_indices_a = [0, len(stats_adv)//4, len(stats_adv)//2, -1]
vis_grids_adv = [grids_adv[i] for i in vis_indices_a]
vis_epochs_adv = [stats_adv[i]["epoch"] for i in vis_indices_a]

plot_epoch_snapshots(
    vis_grids_adv, X_train, y_train, vis_epochs_adv,
    run_name="adversarial", figure_dir=config_base.figure_dir
)
display(Image(filename=os.path.join(config_base.figure_dir, "snapshots_adversarial.png")))

In [ ]:
# Boundary distance histograms at final epoch
plot_boundary_distance_histograms(
    stats_std, stats_adv, epoch_idx=-1, figure_dir=config_base.figure_dir
)
ep_s = stats_std[-1]["epoch"]
ep_a = stats_adv[-1]["epoch"]
display(Image(filename=os.path.join(config_base.figure_dir, f"boundary_dist_hist_ep{ep_s}.png")))

## 11. Poisson baseline comparison

In [ ]:
# Compare learned tessellation vs matched-intensity Poisson baseline
# Use training progress (0 to 1) as x-axis since epochs differ
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

progress_s = [s["epoch"] / STD_EPOCHS for s in stats_std]
progress_a = [s["epoch"] / ADV_EPOCHS for s in stats_adv]

# (a) Region count: learned vs Poisson
ax = axes[0]
ax.plot(progress_s, [s["num_regions_grid"] for s in stats_std],
        'o-', label="Standard", color='#2196F3', markersize=3)
ax.plot(progress_a, [s["num_regions_grid"] for s in stats_adv],
        's-', label="Adversarial", color='#FF5722', markersize=3)
ax.plot(progress_s, [s.get("poisson_num_regions", 0) for s in stats_std],
        '--', label="Poisson (std)", color='#2196F3', alpha=0.5)
ax.plot(progress_a, [s.get("poisson_num_regions", 0) for s in stats_adv],
        '--', label="Poisson (adv)", color='#FF5722', alpha=0.5)
ax.set_xlabel("Training Progress"); ax.set_ylabel("# Regions")
ax.set_title("Region Count: Learned vs Poisson"); ax.legend(fontsize=8)

# (b) Cell area median
ax = axes[1]
ax.plot(progress_s, [s.get("cell_area_median", 0) for s in stats_std],
        'o-', label="Standard", color='#2196F3', markersize=3)
ax.plot(progress_a, [s.get("cell_area_median", 0) for s in stats_adv],
        's-', label="Adversarial", color='#FF5722', markersize=3)
ax.plot(progress_s, [s.get("poisson_cell_area_median", 0) for s in stats_std],
        '--', label="Poisson (std)", color='#2196F3', alpha=0.5)
ax.plot(progress_a, [s.get("poisson_cell_area_median", 0) for s in stats_adv],
        '--', label="Poisson (adv)", color='#FF5722', alpha=0.5)
ax.set_xlabel("Training Progress"); ax.set_ylabel("Median Cell Area")
ax.set_title("Cell Area: Learned vs Poisson"); ax.legend(fontsize=8)

# (c) Boundary density
ax = axes[2]
ax.plot(progress_s, [s.get("boundary_density", 0) for s in stats_std],
        'o-', label="Standard", color='#2196F3', markersize=3)
ax.plot(progress_a, [s.get("boundary_density", 0) for s in stats_adv],
        's-', label="Adversarial", color='#FF5722', markersize=3)
ax.plot(progress_s, [s.get("poisson_boundary_density", 0) for s in stats_std],
        '--', label="Poisson (std)", color='#2196F3', alpha=0.5)
ax.plot(progress_a, [s.get("poisson_boundary_density", 0) for s in stats_adv],
        '--', label="Poisson (adv)", color='#FF5722', alpha=0.5)
ax.set_xlabel("Training Progress"); ax.set_ylabel("Boundary Density")
ax.set_title("Boundary Density: Learned vs Poisson"); ax.legend(fontsize=8)

plt.suptitle(f"Poisson Baseline Comparison ({DATASET})", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(config_base.figure_dir, "poisson_comparison.png"), dpi=200)
plt.savefig(os.path.join(config_base.figure_dir, "poisson_comparison.pdf"))
plt.show()

## 12. Save results

In [ ]:
def serialize_stats(stats_list):
    """Convert stats to JSON-serializable format."""
    out = []
    for s in stats_list:
        d = {}
        for k, v in s.items():
            if isinstance(v, np.ndarray):
                d[k] = v.tolist()
            elif isinstance(v, (np.float32, np.float64)):
                d[k] = float(v)
            elif isinstance(v, (np.int32, np.int64)):
                d[k] = int(v)
            else:
                try:
                    json.dumps(v)
                    d[k] = v
                except (TypeError, ValueError):
                    pass
        out.append(d)
    return out

results = {
    "config": {
        "dataset": DATASET,
        "standard": {
            "hidden_dims": STD_HIDDEN_DIMS,
            "epochs": STD_EPOCHS,
            "lr": STD_LR,
            "scheduler": "none",
        },
        "adversarial": {
            "hidden_dims": ADV_HIDDEN_DIMS,
            "epochs": ADV_EPOCHS,
            "lr": ADV_LR,
            "scheduler": ADV_SCHEDULER,
            "epsilon": ADV_EPSILON,
            "step_size": ADV_EPSILON / 4,
            "pgd_steps": 7,
            "norm": "l2",
        },
        "seed": SEED,
    },
    "convergence": {
        "standard_final_test_acc": final_test_std,
        "adversarial_final_test_acc": final_test_adv,
        "adversarial_final_robust_acc": final_robust_adv,
        "accuracy_gap": abs(final_test_std - final_test_adv),
        "converged": abs(final_test_std - final_test_adv) <= 0.02,
    },
    "standard": serialize_stats(stats_std),
    "adversarial": serialize_stats(stats_adv),
}

results_path = os.path.join(config_base.results_dir, "results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {results_path}")
print(f"Figures saved to {config_base.figure_dir}/")

In [ ]:
# Download results (optional)
from google.colab import files

# Zip everything
!cd /content && zip -r experiment_results.zip \
    figures_{DATASET}/ results_{DATASET}/ checkpoints_{DATASET}/ \
    -x '*.pyc' '__pycache__/*'

files.download("/content/experiment_results.zip")

## 13. Hyperparameter sweep (if convergence failed)

If the adversarial model did not converge to within 2% of the standard model's
clean test accuracy, try the sweep below. Each row tests a different combination
of (width, epsilon, learning_rate, epochs) to find configurations that converge.

In [ ]:
# # ---- Adversarial convergence sweep ----
# # Uncomment to run. Tests different combos to find what converges.
#
# SWEEP_CONFIGS = [
#     # (hidden_dims, epsilon, lr, scheduler, epochs)
#     ([100], 0.05, 0.1, "cosine", 1000),
#     ([200], 0.05, 0.1, "cosine", 1500),
#     ([200], 0.03, 0.1, "cosine", 1000),
#     ([200], 0.1,  0.1, "cosine", 2000),
#     ([100, 100], 0.05, 0.1, "cosine", 1500),
# ]
#
# sweep_results = []
# for hdims, eps, lr, sched, ep in SWEEP_CONFIGS:
#     name = f"adv_{'x'.join(map(str, hdims))}_eps{eps}_lr{lr}_{sched}_{ep}ep"
#     print(f"\n--- {name} ---")
#     cfg = make_config(
#         dataset=DATASET, hidden_dims=hdims, epochs=ep,
#         epsilon=eps, lr=lr, scheduler=sched,
#         adversarial=True, seed=SEED,
#         device="cuda" if torch.cuda.is_available() else "cpu",
#     )
#     cfg.checkpoint_dir = config_base.checkpoint_dir
#     cfg.tess.checkpoint_epochs = []  # skip checkpoints for sweep
#
#     model = make_relu_mlp(2, hdims, 2)
#     dataloader = get_dataloader(dataset, batch_size=64)
#     hist = train_model(
#         model, dataloader, cfg,
#         checkpoint_dir=os.path.join(cfg.checkpoint_dir, name),
#         run_name=name, device=cfg.device,
#         X_test=X_test, y_test=y_test,
#     )
#     final_clean = [a for a in hist["test_clean_acc"] if a > 0][-1]
#     final_robust = [a for a in hist["test_robust_acc"] if a > 0][-1] if any(
#         a > 0 for a in hist["test_robust_acc"]) else 0.0
#
#     sweep_results.append({
#         "name": name, "hidden_dims": hdims, "epsilon": eps,
#         "lr": lr, "scheduler": sched, "epochs": ep,
#         "final_clean_test_acc": final_clean,
#         "final_robust_test_acc": final_robust,
#         "gap_vs_standard": abs(final_test_std - final_clean),
#     })
#     print(f"  Clean: {final_clean:.4f}, Robust: {final_robust:.4f}, "
#           f"Gap: {abs(final_test_std - final_clean):.4f}")
#
# # Summary table
# print(f"\n{'='*80}")
# print(f"{'Config':<50} {'Clean':>8} {'Robust':>8} {'Gap':>8}")
# print(f"{'='*80}")
# for r in sorted(sweep_results, key=lambda x: x["gap_vs_standard"]):
#     print(f"{r['name']:<50} {r['final_clean_test_acc']:>8.4f} "
#           f"{r['final_robust_test_acc']:>8.4f} {r['gap_vs_standard']:>8.4f}")
# print(f"Standard model reference:{'':>25} {final_test_std:>8.4f}")